In [1]:
import sys
import importlib
sys.path.append('../')  # Adjust the path as needed

import utilities.functions as functions
import utilities.plot as plot

# Reload the module to reflect the changes
importlib.reload(functions)
importlib.reload(plot)

<module 'utilities.plot' from '/Users/xuechenkan/potts_model_test/ms0.5/../utilities/plot.py'>

In [2]:
IN_seq_path = 'IN/data/in.reduce4.seq'
PR_seq_path = 'PR/data/pr.exper.reduce4.seq'    
RT_seq_path = 'RT/data/rt.reduce4.seq'

IN_all_seq = functions.read_seq('IN/data/in.reduce4.seq')
PR_all_seq = functions.read_seq('PR/data/pr.exper.reduce4.seq')
RT_all_seq = functions.read_seq('RT/data/rt.reduce4.seq')

IN_consensus = 'IN/data/in.consensus.reduce4.seq'
with open(IN_consensus, 'r') as f:
    IN_consensus_seq = f.read().strip()
# print("IN consensus sequence:", IN_consensus_seq)
PR_consensus = 'PR/data/pr.consensus.reduce4.seq'
with open(PR_consensus, 'r') as f:
    PR_consensus_seq = f.read().strip()
RT_consensus = 'RT/data/rt.consensus.reduce4.seq'
with open(RT_consensus, 'r') as f:
    RT_consensus_seq = f.read().strip()

# print(IN_consensus_seq)
IN_min_position = 1
IN_max_position = 263
PR_min_position = 1
PR_max_position = 99
RT_min_position = 39
RT_max_position = 226

IN_redux = functions.get_redu_dict('IN/data/in.reduce4.redux',1)
PR_redux = functions.get_redu_dict('PR/data/pr.reduce4.redux',0)
RT_redux = functions.get_redu_dict('RT/data/rt.reduce4.redux',0)

IN_J = functions.load_J_dict('IN/data/J.npy',IN_min_position,IN_max_position)
PR_J = functions.load_J_dict('PR/data/J_PR.npy',PR_min_position,PR_max_position)
RT_J = functions.load_J_dict('RT/data/J_RT.npy',RT_min_position,RT_max_position)

IN_all_seq_unreduced = functions.read_seq('IN/data/in.fullseq')
PR_all_seq_unreduced = functions.read_seq('PR/data/pr.exper.fullseq')
RT_all_seq_unreduced = functions.read_seq('RT/data/rt.fullseq')

In [3]:
from tqdm import tqdm
import pandas as pd
IN_de_dict = {}
IN_de_reduced_to_unreduced = {}
IN_dde = {}
IN_dde_unreduced = {}
for i in tqdm(range(0, IN_max_position-IN_min_position+1), desc="IN Progress"):
    pos1 = i + 1
    wt1 = IN_consensus_seq[i]
    for aa in ['A', 'B', 'C', 'D']:
        if aa == wt1:
            continue
        mutation_pair1 = wt1 + str(pos1) + aa
        mutation_pair1_unreduced = functions.reduced_to_unreduced(
            IN_redux, mutation_pair1, IN_all_seq_unreduced, IN_min_position
        )
        de = functions.calculate_delta_e(mutation_pair1, IN_consensus_seq, IN_J, IN_min_position, IN_max_position)
        IN_de_dict[mutation_pair1] = float(de)
        IN_de_reduced_to_unreduced[mutation_pair1] = mutation_pair1_unreduced
# IN:
for i in tqdm(range(0, IN_max_position-IN_min_position+1), desc="IN Progress"):
    pos1 = i+1
    wt1 = IN_consensus_seq[i]
    for aa in ['A','B','C','D']:
        if aa == wt1:
            continue
        mutation_pair1 = wt1 + str(pos1) + aa
        for j in range(i+1, IN_max_position-IN_min_position+1):
            pos2 = j+1
            wt2 = IN_consensus_seq[j]
            for aa in ['A','B','C','D']:
                if aa == wt2:
                    continue
                mutation_pair2 = wt2 + str(pos2) + aa
                de12 = functions.calculate_delta_e_double(mutation_pair1, mutation_pair2, IN_consensus_seq, IN_J, IN_min_position, IN_max_position)
                dde = de12 - IN_de_dict[mutation_pair1] - IN_de_dict[mutation_pair2]
                IN_dde[(mutation_pair1, mutation_pair2)] = float(dde)

                pair1_unreduced = IN_de_reduced_to_unreduced.get(mutation_pair1, mutation_pair1)
                pair2_unreduced = IN_de_reduced_to_unreduced.get(mutation_pair2, mutation_pair2)
                IN_dde_unreduced[(pair1_unreduced, pair2_unreduced)] = float(dde)

pd.DataFrame(
    [(k, v) for k, v in IN_de_dict.items()],
    columns=["mutation", "delta_e"]
).to_csv("IN_de_dict.csv", index=False)

pd.DataFrame(
    [(k, v) for k, v in IN_de_reduced_to_unreduced.items()],
    columns=["mutation", "unreduced_mutation"]
).to_csv("IN_de_reduced_to_unreduced.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in IN_dde.items()],
    columns=["mutation_pair", "dde"]
).to_csv("IN_dde.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in IN_dde_unreduced.items()],
    columns=["mutation_pair", "dde"]
).to_csv("IN_dde_unreduced.csv", index=False)

print("CSV files saved successfully.")

IN Progress: 100%|██████████| 263/263 [02:04<00:00,  2.10it/s]


CSV files saved successfully.


In [4]:
PR_de_dict = {}
PR_de_reduced_to_unreduced = {}
PR_dde = {}
PR_dde_unreduced = {}

# Single-mutation ΔE
for i in tqdm(range(0, PR_max_position - PR_min_position + 1), desc="PR Progress (single)"):
    pos1 = i + PR_min_position
    wt1 = PR_consensus_seq[i]

    for aa in ['A', 'B', 'C', 'D']:
        if aa == wt1:
            continue

        mutation_pair1 = wt1 + str(pos1) + aa
        mutation_pair1_unreduced = functions.reduced_to_unreduced(
            PR_redux, mutation_pair1, PR_all_seq_unreduced, PR_min_position
        )
        de = functions.calculate_delta_e(
            mutation_pair1, PR_consensus_seq, PR_J, PR_min_position, PR_max_position
        )

        PR_de_dict[mutation_pair1] = float(de)
        PR_de_reduced_to_unreduced[mutation_pair1] = mutation_pair1_unreduced

# Double-mutation ΔΔE
for i in tqdm(range(0, PR_max_position - PR_min_position + 1), desc="PR Progress (double)"):
    pos1 = i + PR_min_position
    wt1 = PR_consensus_seq[i]

    for aa1 in ['A', 'B', 'C', 'D']:
        if aa1 == wt1:
            continue

        mutation_pair1 = wt1 + str(pos1) + aa1

        for j in range(i + 1, PR_max_position - PR_min_position + 1):
            pos2 = j + PR_min_position
            wt2 = PR_consensus_seq[j]

            for aa2 in ['A', 'B', 'C', 'D']:
                if aa2 == wt2:
                    continue

                mutation_pair2 = wt2 + str(pos2) + aa2
                de12 = functions.calculate_delta_e_double(
                    mutation_pair1, mutation_pair2, PR_consensus_seq, PR_J, PR_min_position, PR_max_position
                )
                dde = de12 - PR_de_dict[mutation_pair1] - PR_de_dict[mutation_pair2]

                PR_dde[(mutation_pair1, mutation_pair2)] = float(dde)

                pair1_unreduced = PR_de_reduced_to_unreduced.get(mutation_pair1, mutation_pair1)
                pair2_unreduced = PR_de_reduced_to_unreduced.get(mutation_pair2, mutation_pair2)
                PR_dde_unreduced[(pair1_unreduced, pair2_unreduced)] = float(dde)

# Keep compatibility with downstream cells (e.g., pickle dump)
PR_de_results = PR_de_dict
PR_dde_results = PR_dde

# Save CSVs
pd.DataFrame(
    [(k, v) for k, v in PR_de_dict.items()],
    columns=["mutation", "delta_e"]
).to_csv("PR_de_dict.csv", index=False)

pd.DataFrame(
    [(k, v) for k, v in PR_de_reduced_to_unreduced.items()],
    columns=["mutation", "unreduced_mutation"]
).to_csv("PR_de_reduced_to_unreduced.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in PR_dde.items()],
    columns=["mutation_pair", "dde"]
).to_csv("PR_dde.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in PR_dde_unreduced.items()],
    columns=["mutation_pair", "dde"]
).to_csv("PR_dde_unreduced.csv", index=False)

print("PR CSV files saved successfully.")

PR Progress (double): 100%|██████████| 99/99 [00:05<00:00, 16.73it/s]


PR CSV files saved successfully.


In [5]:
RT_de_dict = {}
RT_de_reduced_to_unreduced = {}
RT_dde = {}
RT_dde_unreduced = {}

# Single-mutation ΔE
for i in tqdm(range(0, RT_max_position - RT_min_position + 1), desc="RT Progress (single)"):
    pos1 = i + RT_min_position
    wt1 = RT_consensus_seq[i]

    for aa1 in ['A', 'B', 'C', 'D']:
        if aa1 == wt1:
            continue

        mutation_pair1 = wt1 + str(pos1) + aa1
        mutation_pair1_unreduced = functions.reduced_to_unreduced(
            RT_redux, mutation_pair1, RT_all_seq_unreduced, RT_min_position
        )
        de = functions.calculate_delta_e(
            mutation_pair1, RT_consensus_seq, RT_J, RT_min_position, RT_max_position
        )

        RT_de_dict[mutation_pair1] = float(de)
        RT_de_reduced_to_unreduced[mutation_pair1] = mutation_pair1_unreduced

# Double-mutation ΔΔE
for i in tqdm(range(0, RT_max_position - RT_min_position + 1), desc="RT Progress (double)"):
    pos1 = i + RT_min_position
    wt1 = RT_consensus_seq[i]

    for aa1 in ['A', 'B', 'C', 'D']:
        if aa1 == wt1:
            continue

        mutation_pair1 = wt1 + str(pos1) + aa1

        for j in range(i + 1, RT_max_position - RT_min_position + 1):
            pos2 = j + RT_min_position
            wt2 = RT_consensus_seq[j]

            for aa2 in ['A', 'B', 'C', 'D']:
                if aa2 == wt2:
                    continue

                mutation_pair2 = wt2 + str(pos2) + aa2
                de12 = functions.calculate_delta_e_double(
                    mutation_pair1, mutation_pair2, RT_consensus_seq, RT_J, RT_min_position, RT_max_position
                )
                dde = de12 - RT_de_dict[mutation_pair1] - RT_de_dict[mutation_pair2]

                RT_dde[(mutation_pair1, mutation_pair2)] = float(dde)

                pair1_unreduced = RT_de_reduced_to_unreduced.get(mutation_pair1, mutation_pair1)
                pair2_unreduced = RT_de_reduced_to_unreduced.get(mutation_pair2, mutation_pair2)
                RT_dde_unreduced[(pair1_unreduced, pair2_unreduced)] = float(dde)

# Keep compatibility with downstream pickle cell
RT_de_results = RT_de_dict
RT_dde_results = RT_dde

# Save CSVs
pd.DataFrame(
    [(k, v) for k, v in RT_de_dict.items()],
    columns=["mutation", "delta_e"]
).to_csv("RT_de_dict.csv", index=False)

pd.DataFrame(
    [(k, v) for k, v in RT_de_reduced_to_unreduced.items()],
    columns=["mutation", "unreduced_mutation"]
).to_csv("RT_de_reduced_to_unreduced.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in RT_dde.items()],
    columns=["mutation_pair", "dde"]
).to_csv("RT_dde.csv", index=False)

pd.DataFrame(
    [(str(k), v) for k, v in RT_dde_unreduced.items()],
    columns=["mutation_pair", "dde"]
).to_csv("RT_dde_unreduced.csv", index=False)

print("RT CSV files saved successfully.")

RT Progress (double): 100%|██████████| 188/188 [00:46<00:00,  4.04it/s]


RT CSV files saved successfully.


In [18]:
import ast
import re

# Load DDE CSV (accept either filename)
csv_candidates = ["IN_dde_dict.csv", "IN_dde.csv"]
for csv_file in csv_candidates:
    try:
        in_dde_df = pd.read_csv(csv_file)
        print(f"Loaded: {csv_file}")
        break
    except FileNotFoundError:
        in_dde_df = None

if in_dde_df is None:
    raise FileNotFoundError("Neither IN_dde_dict.csv nor IN_dde.csv was found.")

# Sort by highest DDE
in_dde_df = in_dde_df.sort_values("dde", ascending=False).reset_index(drop=True)

# Get top mutation pair
top_pair_raw = in_dde_df.loc[0, "mutation_pair"]
top_dde = in_dde_df.loc[0, "dde"]
m1_red, m2_red = ast.literal_eval(top_pair_raw)

# Convert reduced mutation labels -> unreduced mutation labels if mapping exists
m1 = IN_de_reduced_to_unreduced.get(m1_red, m1_red)
m2 = IN_de_reduced_to_unreduced.get(m2_red, m2_red)

def parse_mutation(mut):
    # format: W123M
    m = re.match(r"^([A-Z])(\d+)([A-Z])$", mut)
    if not m:
        raise ValueError(f"Cannot parse mutation string: {mut}")
    wt, pos, mt = m.groups()
    return wt, int(pos), mt

def single_mutant_frequency(seqs, mutation,min_pos = 1):
    _, pos, mt = parse_mutation(mutation)
    idx = pos - min_pos  # positions are 1-based
    count = sum(1 for s in seqs if len(s) > idx and s[idx] == mt)
    return count / len(seqs)

def double_mutant_frequency(seqs, mutation1, mutation2, min_pos = 1):
    _, pos1, mt1 = parse_mutation(mutation1)
    _, pos2, mt2 = parse_mutation(mutation2)
    i1, i2 = pos1 - min_pos, pos2 - min_pos
    count = sum(1 for s in seqs if len(s) > max(i1, i2) and s[i1] == mt1 and s[i2] == mt2)
    return count / len(seqs)

def build_dde_summary(df_subset):
    records = []

    for _, row in df_subset.iterrows():
        pair_raw = row["mutation_pair"]
        dde_val = float(row["dde"])

        # Parse reduced pair from CSV string like "('C140D', 'D148B')"
        m1_red, m2_red = ast.literal_eval(pair_raw)

        # Map reduced -> unreduced if available
        m1_unred = IN_de_reduced_to_unreduced.get(m1_red, m1_red)
        m2_unred = IN_de_reduced_to_unreduced.get(m2_red, m2_red)

        de1 = float(IN_de_dict.get(m1_red, float("nan")))
        de2 = float(IN_de_dict.get(m2_red, float("nan")))

        f1 = single_mutant_frequency(IN_all_seq_unreduced, m1_unred)
        f2 = single_mutant_frequency(IN_all_seq_unreduced, m2_unred)
        f12 = double_mutant_frequency(IN_all_seq_unreduced, m1_unred, m2_unred)

        records.append({
            "double mutation pair": f"{m1_unred},{m2_unred}",
            "single mutaiton pair 1": m1_unred,
            "single mutation pair 2": m2_unred,
            "de1": de1,
            "de2": de2,
            "dde": dde_val,
            "freq mutation 1": f1,
            "freq mutation 2": f2,
            "freq double mutation": f12,
        })

    return pd.DataFrame(records)
n = 1000
top_150_df = build_dde_summary(in_dde_df.nlargest(n, "dde"))
bottom_150_df = build_dde_summary(in_dde_df.nsmallest(n, "dde"))

top_150_df.to_csv(f"IN_top_{n}_dde_summary.csv", index=False)
bottom_150_df.to_csv(f"IN_bottom_{n}_dde_summary.csv", index=False)

print(f"Saved: IN_top_{n}_dde_summary.csv")
print(f"Saved: IN_bottom_{n}_dde_summary.csv")

Loaded: IN_dde.csv
Saved: IN_top_1000_dde_summary.csv
Saved: IN_bottom_1000_dde_summary.csv


In [17]:
# Load DDE CSV (accept either filename)
csv_candidates = ["PR_dde_dict.csv", "PR_dde.csv"]
for csv_file in csv_candidates:
    try:
        pr_dde_df = pd.read_csv(csv_file)
        print(f"Loaded: {csv_file}")
        break
    except FileNotFoundError:
        pr_dde_df = None

if pr_dde_df is None:
    raise FileNotFoundError("Neither PR_dde_dict.csv nor PR_dde.csv was found.")

# Sort by highest DDE
pr_dde_df = pr_dde_df.sort_values("dde", ascending=False).reset_index(drop=True)

# Get top mutation pair
top_pair_raw = pr_dde_df.loc[0, "mutation_pair"]
top_dde = pr_dde_df.loc[0, "dde"]
m1_red, m2_red = ast.literal_eval(top_pair_raw)

# Convert reduced mutation labels -> unreduced mutation labels if mapping exists
m1 = PR_de_reduced_to_unreduced.get(m1_red, m1_red)
m2 = PR_de_reduced_to_unreduced.get(m2_red, m2_red)

def build_pr_dde_summary(df_subset):
    records = []

    for _, row in df_subset.iterrows():
        pair_raw = row["mutation_pair"]
        dde_val = float(row["dde"])

        # Parse reduced pair from CSV string like "('C30D', 'D88B')"
        m1_red, m2_red = ast.literal_eval(pair_raw)

        # Map reduced -> unreduced if available
        m1_unred = PR_de_reduced_to_unreduced.get(m1_red, m1_red)
        m2_unred = PR_de_reduced_to_unreduced.get(m2_red, m2_red)

        de1 = float(PR_de_dict.get(m1_red, float("nan")))
        de2 = float(PR_de_dict.get(m2_red, float("nan")))

        f1 = single_mutant_frequency(PR_all_seq_unreduced, m1_unred)
        f2 = single_mutant_frequency(PR_all_seq_unreduced, m2_unred)
        f12 = double_mutant_frequency(PR_all_seq_unreduced, m1_unred, m2_unred)

        records.append({
            "double mutation pair": f"{m1_unred},{m2_unred}",
            "single mutaiton pair 1": m1_unred,
            "single mutation pair 2": m2_unred,
            "de1": de1,
            "de2": de2,
            "dde": dde_val,
            "freq mutation 1": f1,
            "freq mutation 2": f2,
            "freq double mutation": f12,
        })

    return pd.DataFrame(records)
n = 1000
PR_top_150_df = build_pr_dde_summary(pr_dde_df.nlargest(n, "dde"))
PR_bottom_150_df = build_pr_dde_summary(pr_dde_df.nsmallest(n, "dde"))

PR_top_150_df.to_csv(f"PR_top_{n}_dde_summary.csv", index=False)
PR_bottom_150_df.to_csv(f"PR_bottom_{n}_dde_summary.csv", index=False)

print(f"Saved: PR_top_{n}_dde_summary.csv")
print(f"Saved: PR_bottom_{n}_dde_summary.csv")
print(f"Top PR pair: {m1}, {m2} (DDE={top_dde:.6f})")

Loaded: PR_dde.csv
Saved: PR_top_1000_dde_summary.csv
Saved: PR_bottom_1000_dde_summary.csv
Top PR pair: P1H, Q2T (DDE=7.232627)


In [16]:
# Load DDE CSV (accept either filename)
csv_candidates = ["RT_dde_dict.csv", "RT_dde.csv"]
for csv_file in csv_candidates:
    try:
        rt_dde_df = pd.read_csv(csv_file)
        print(f"Loaded: {csv_file}")
        break
    except FileNotFoundError:
        rt_dde_df = None

if rt_dde_df is None:
    raise FileNotFoundError("Neither RT_dde_dict.csv nor RT_dde.csv was found.")

# Sort by highest DDE
rt_dde_df = rt_dde_df.sort_values("dde", ascending=False).reset_index(drop=True)

# Get top mutation pair
top_pair_raw = rt_dde_df.loc[0, "mutation_pair"]
top_dde = rt_dde_df.loc[0, "dde"]
m1_red, m2_red = ast.literal_eval(top_pair_raw)

# Convert reduced mutation labels -> unreduced mutation labels if mapping exists
m1 = RT_de_reduced_to_unreduced.get(m1_red, m1_red)
m2 = RT_de_reduced_to_unreduced.get(m2_red, m2_red)

def build_rt_dde_summary(df_subset):
    records = []

    for _, row in df_subset.iterrows():
        pair_raw = row["mutation_pair"]
        dde_val = float(row["dde"])

        # Parse reduced pair from CSV string like "('D39A', 'D40B')"
        m1_red, m2_red = ast.literal_eval(pair_raw)

        # Map reduced -> unreduced if available
        m1_unred = RT_de_reduced_to_unreduced.get(m1_red, m1_red)
        m2_unred = RT_de_reduced_to_unreduced.get(m2_red, m2_red)

        de1 = float(RT_de_dict.get(m1_red, float("nan")))
        de2 = float(RT_de_dict.get(m2_red, float("nan")))

        f1 = single_mutant_frequency(RT_all_seq_unreduced, m1_unred, RT_min_position)
        f2 = single_mutant_frequency(RT_all_seq_unreduced, m2_unred, RT_min_position)
        f12 = double_mutant_frequency(RT_all_seq_unreduced, m1_unred, m2_unred, RT_min_position)

        records.append({
            "double mutation pair": f"{m1_unred},{m2_unred}",
            "single mutaiton pair 1": m1_unred,
            "single mutation pair 2": m2_unred,
            "de1": de1,
            "de2": de2,
            "dde": dde_val,
            "freq mutation 1": f1,
            "freq mutation 2": f2,
            "freq double mutation": f12,
        })

    return pd.DataFrame(records)
n = 1000
RT_top_150_df = build_rt_dde_summary(rt_dde_df.nlargest(n, "dde"))
RT_bottom_150_df = build_rt_dde_summary(rt_dde_df.nsmallest(n, "dde"))

RT_top_150_df.to_csv(f"RT_top_{n}_dde_summary.csv", index=False)
RT_bottom_150_df.to_csv(f"RT_bottom_{n}_dde_summary.csv", index=False)

print(f"Saved: RT_top_{n}_dde_summary.csv")
print(f"Saved: RT_bottom_{n}_dde_summary.csv")
print(f"Top RT pair: {m1}, {m2} (DDE={top_dde:.6f})")

Loaded: RT_dde.csv
Saved: RT_top_1000_dde_summary.csv
Saved: RT_bottom_1000_dde_summary.csv
Top RT pair: F116Y, Q151M (DDE=4.382366)
